# Analyse Exploratoire des Données (EDA)
## Plateforme MLOps de Détection d'Intrusions — Dataset CICIDS2017

**Projet :** Plateforme MLOps de cybersécurité pour la détection d'intrusions
**Dataset :** CICIDS2017 (nettoyé via le pipeline Kafka du projet — `cleaned_logs.csv`)

---

### Objectif de ce notebook

Ce notebook a pour objectif d'explorer les données réseau nettoyées produites par le pipeline du projet, afin de :

1. Évaluer la qualité des données (valeurs manquantes, doublons, colonnes redondantes ou constantes) ;
2. Comprendre la distribution des classes de trafic (trafic normal vs types d'attaques) ;
3. Identifier les variables (features) les plus discriminantes entre trafic bénin et trafic malveillant ;
4. Formuler des recommandations pour l'étape de modélisation (Machine Learning) à venir.

Ce notebook accompagne le rapport d'analyse exploratoire remis séparément, et en reprend la structure.

## 1. Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Options d'affichage
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)
plt.rcParams["figure.dpi"] = 100

print("Bibliothèques chargées avec succès.")

## 2. Chargement des données

Le fichier `cleaned_logs.csv` est le résultat du pipeline de nettoyage du projet
(ingestion → publication Kafka → nettoyage). Il contient l'intégralité du trafic
réseau nettoyé, au format CICIDS2017 standardisé (colonnes en `snake_case`).

> **Remarque :** ce fichier étant volumineux (plusieurs millions de lignes), le
> chargement complet peut prendre un certain temps selon la machine utilisée.
> Si la mémoire disponible est insuffisante, il est possible de charger un
> échantillon en décommentant la ligne prévue à cet effet ci-dessous.

In [ ]:
DATA_PATH = "cleaned_logs.csv"  # adapter le chemin si nécessaire

# Chargement complet du dataset nettoyé
df = pd.read_csv(DATA_PATH, low_memory=False)

# Alternative en cas de contrainte mémoire : charger un échantillon aléatoire
# df = pd.read_csv(DATA_PATH, low_memory=False).sample(n=200_000, random_state=42)

print(f"Dimensions du dataset : {df.shape[0]:,} lignes, {df.shape[1]} colonnes")

## 3. Aperçu général des données

In [ ]:
df.head()

In [ ]:
df.info(memory_usage="deep")

In [ ]:
# Statistiques descriptives générales sur les colonnes numériques
df.describe().T

## 4. Qualité des données

Cette section vérifie la complétude et la cohérence du dataset : valeurs
manquantes, doublons, colonnes redondantes ou sans valeur informative
(constantes).

### 4.1 Valeurs manquantes

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if missing.empty:
    print("Aucune valeur manquante détectée dans le dataset.")
else:
    print(missing)

### 4.2 Lignes dupliquées

In [ ]:
n_duplicates = df.duplicated().sum()
pct_duplicates = 100 * n_duplicates / len(df)

print(f"Nombre de lignes dupliquées : {n_duplicates:,} ({pct_duplicates:.2f}% du dataset)")

### 4.3 Colonnes constantes

Une colonne "constante" ne prend qu'une seule valeur sur l'ensemble du dataset :
elle n'apporte donc aucune information discriminante et peut être écartée avant
la modélisation.

In [ ]:
constant_cols = [c for c in df.columns if df[c].nunique() == 1]

print(f"Nombre de colonnes constantes : {len(constant_cols)}")
print(constant_cols)

### 4.4 Colonnes redondantes

Vérification de la présence de colonnes dupliquées suite au nettoyage
(par exemple `fwd_header_length` et `fwd_header_length_1`, identifiées lors
d'une première exploration du pipeline).

In [ ]:
if "fwd_header_length" in df.columns and "fwd_header_length_1" in df.columns:
    identical = (df["fwd_header_length"] == df["fwd_header_length_1"]).all()
    print(f"fwd_header_length et fwd_header_length_1 sont strictement identiques : {identical}")
else:
    print("Les colonnes fwd_header_length / fwd_header_length_1 ne sont pas toutes deux présentes.")

### 4.5 Synthèse des recommandations de nettoyage complémentaire

D'après les constats ci-dessus, les actions suivantes sont recommandées avant
la phase de modélisation :

- Suppression des lignes dupliquées ;
- Suppression des colonnes constantes (sans pouvoir discriminant) ;
- Suppression de la colonne redondante `fwd_header_length_1` ;
- Correction, en amont dans le pipeline, de l'encodage de lecture des fichiers
  CSV bruts (des caractères spéciaux dans les libellés `label` apparaissent
  actuellement mal encodés).

## 5. Distribution des classes de trafic

Cette section examine la répartition du trafic entre les différentes classes
(`label`) ainsi qu'entre trafic bénin et trafic d'attaque (`is_attack`).

In [ ]:
label_counts = df["label"].astype(str).str.strip().value_counts()
label_pct = 100 * label_counts / len(df)

summary_labels = pd.DataFrame({"nb_lignes": label_counts, "pourcentage": label_pct.round(3)})
summary_labels

In [ ]:
if "is_attack" in df.columns:
    attack_counts = df["is_attack"].value_counts().sort_index()
    attack_pct = 100 * attack_counts / len(df)
    print("Répartition bénin (0) / attaque (1) :")
    for idx, count in attack_counts.items():
        label_name = "Bénin" if idx == 0 else "Attaque"
        print(f"  {label_name:10s} : {count:>10,} ({attack_pct[idx]:.2f}%)")

**Figure 1 — Distribution des classes de trafic**

Lecture : chaque barre représente le nombre de lignes appartenant à une classe
de trafic donnée. Une échelle logarithmique est utilisée en abscisse car les
effectifs varient de quelques unités (classes d'attaque rares) à plusieurs
millions de lignes (trafic bénin).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
order = label_counts.index
ax.barh(order, label_counts.values, color="#4C72B0")
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xlabel("Nombre de lignes (échelle logarithmique)")
ax.set_title(f"Distribution des classes de trafic (n={len(df):,})")
plt.tight_layout()
plt.show()

**Interprétation :** le dataset présente un déséquilibre très marqué entre
classes, caractéristique des jeux de données de détection d'intrusion réseau.
Le trafic bénin est très largement majoritaire, tandis que certaines classes
d'attaque (par exemple `Heartbleed`, `Infiltration`, `Web Attack – SQL
Injection`) sont extrêmement rares. Ce déséquilibre devra être pris en compte
lors de la modélisation (rééquilibrage des classes, métriques adaptées).

## 6. Analyse des features clés

Cette section compare les principales caractéristiques de flux réseau entre
trafic bénin et trafic d'attaque, puis étudie les corrélations entre ces
variables.

In [ ]:
KEY_FEATURES = [
    "flow_duration",
    "total_fwd_packets",
    "total_backward_packets",
    "flow_bytes_per_s",
    "flow_packets_per_s",
    "average_packet_size",
    "fwd_packet_length_mean",
    "bwd_packet_length_mean",
]

KEY_FEATURES = [f for f in KEY_FEATURES if f in df.columns]
KEY_FEATURES

### 6.1 Statistiques descriptives par classe (bénin vs attaque)

In [ ]:
if "is_attack" in df.columns:
    stats_by_class = df.groupby("is_attack")[KEY_FEATURES].mean().T
    stats_by_class.columns = ["Bénin", "Attaque"]
    stats_by_class

**Figure 2 — Durée des flux réseau selon le type de trafic**

Lecture : l'axe horizontal représente la durée des flux (`flow_duration`) sur
une échelle logarithmique en base 10, afin de pouvoir comparer des durées très
courtes et très longues sur un même graphique.

In [ ]:
if "flow_duration" in df.columns and "is_attack" in df.columns:
    fig, ax = plt.subplots(figsize=(9, 6))
    log_fd = np.log10(df["flow_duration"].clip(lower=1))

    for val, color, name in [(0, "#55A868", "Bénin"), (1, "#C44E52", "Attaque")]:
        data = log_fd[df["is_attack"] == val]
        ax.hist(data, bins=45, alpha=0.6, label=name, color=color)

    ax.set_xlabel("log10(flow_duration)")
    ax.set_ylabel("Nombre de flux")
    ax.set_title("Distribution de flow_duration — bénin vs attaque")
    ax.legend()
    plt.tight_layout()
    plt.show()

### 6.2 Matrice de corrélation

> **Remarque méthodologique :** la variable `flow_duration` présente une
> amplitude de valeurs très élevée et hétérogène. Sur de très gros volumes de
> données, son écart-type et ses corrélations peuvent devenir instables
> numériquement. Il est recommandé, avant toute utilisation en modélisation,
> d'appliquer une transformation (par exemple logarithmique) à cette variable.

In [ ]:
corr_matrix = df[KEY_FEATURES].corr()
corr_matrix.round(3)

**Figure 3 — Corrélations entre les features réseau clés**

Lecture : chaque case indique le coefficient de corrélation entre deux
variables (de -1 à +1). Le rouge foncé signale une forte corrélation positive,
le bleu une corrélation négative, le blanc/gris une absence de lien
linéaire.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(KEY_FEATURES)))
ax.set_yticks(range(len(KEY_FEATURES)))
ax.set_xticklabels(KEY_FEATURES, rotation=90)
ax.set_yticklabels(KEY_FEATURES)
ax.set_title("Corrélation entre features clés")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

**Interprétation :** deux redondances fortes se dégagent : `total_fwd_packets`
avec `total_backward_packets`, et `average_packet_size` avec
`bwd_packet_length_mean`. Ces paires de variables apportent une information
très similaire ; n'en conserver qu'une par paire permettrait de simplifier un
futur modèle sans perte d'information significative.

### 6.3 Taille des paquets par classe d'attaque

**Figure 4 — Taille moyenne des paquets selon le type d'attaque**

Lecture : ce diagramme en boîte (boxplot) montre, pour les classes les plus
fréquentes, la répartition de la taille moyenne des paquets
(`average_packet_size`). Les valeurs extrêmes (99e percentile) sont écrêtées
pour préserver la lisibilité.

In [ ]:
top_labels = label_counts.head(6).index.tolist()
subset = df[df["label"].astype(str).str.strip().isin(top_labels)]

data_to_plot, present_labels = [], []
for lab in top_labels:
    vals = subset.loc[subset["label"].astype(str).str.strip() == lab, "average_packet_size"].dropna()
    if len(vals) > 0:
        cap = vals.quantile(0.99)
        data_to_plot.append(vals.clip(upper=cap))
        present_labels.append(lab)

fig, ax = plt.subplots(figsize=(9, 6))
ax.boxplot(data_to_plot, tick_labels=present_labels, vert=False)
ax.set_xlabel("average_packet_size")
ax.set_title("average_packet_size par classe (top 6 classes les plus fréquentes)")
plt.tight_layout()
plt.show()

**Interprétation :** les attaques de type `DDoS` et `DoS Hulk` se distinguent
par des paquets nettement plus volumineux et plus variables que le trafic
bénin, tandis que des attaques comme `PortScan` ou `FTP-Patator` présentent au
contraire des paquets très petits et homogènes. Cela confirme le pouvoir
discriminant de cette variable pour la détection d'intrusions.

## 7. Synthèse et recommandations

**Qualité des données**
- Compléter le nettoyage réalisé par le pipeline (`cleaner.py`) en supprimant
  les doublons, la colonne redondante et les colonnes constantes identifiées
  en section 4.
- Corriger l'encodage de lecture des fichiers bruts pour restaurer les
  libellés d'attaque corrects.

**Déséquilibre des classes**
- Le trafic bénin représente la grande majorité des lignes, contre une part
  très réduite pour certaines classes d'attaque rares. Une stratégie de
  rééquilibrage (sur-échantillonnage type SMOTE, pondération de classes, ou
  approche par détection d'anomalies) sera nécessaire avant modélisation.

**Feature engineering**
- `average_packet_size`, `bwd_packet_length_mean` et `flow_duration`
  apparaissent comme les variables les plus discriminantes entre trafic bénin
  et trafic d'attaque.
- Les redondances observées (`total_fwd_packets` / `total_backward_packets` ;
  `average_packet_size` / `bwd_packet_length_mean`) invitent à une sélection
  de variables avant l'entraînement d'un modèle.

**Prochaines étapes**
- Partager ces constats avec l'équipe Data Engineering / Data Quality
  (nettoyage complémentaire, correction d'encodage).
- Transmettre la synthèse des features discriminantes à l'équipe Machine
  Learning pour orienter la modélisation.